# E05b: SASRec на GPU (удалённое ядро)

Самодостаточный ноутбук для запуска на GPU-ядре вне репозитория: воспроизводит
пайплайн `scripts/e05_sasrec.py` с большей ёмкостью (embedding 192, окно 200,
40 эпох), недоступной нашему CPU за разумное время.

**Протокол идентичен локальному** (notes/teacher_rules.md): обучение только на
train-части сплита, никаких статистик из holdout; recall@10 — те же варианты
делителя, что в `src/recsys/metrics.py`, чтобы числа были сравнимы с E01–E05.

## Подготовка
1. Положите рядом с ноутбуком файл `events.csv` и `item_features.csv`
   (каталог `data/` или рядом — путь настраивается в следующей ячейке);
2. Запустите ячейки по порядку;
3. Результаты: `gpu_outputs/results_e05b.json` и `gpu_outputs/E05b_sasrec.csv`
   (сабмит по полным events) — заберите оба файла в репозиторий:
   json → `results/`, csv → `submissions/`, и запишите эксперимент в журнал.

In [ ]:
%pip install -q replay-rec lightning pyarrow 2>&1 | tail -1

In [ ]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU не виден — проверьте ядро

In [ ]:
from pathlib import Path

DATA_DIR = Path("data") if Path("data/events.csv").is_file() else Path(".")
assert (DATA_DIR / "events.csv").is_file(), "положите events.csv в ./data или рядом с ноутбуком"
for f in ("events.csv", "item_features.csv"):
    print(f, "->", (DATA_DIR / f).stat().st_size // 1024, "KiB")

## Данные, сплиты, метрика (копия локального протокола)

- `chronological_split(0.9)` — основной temporal-протокол;
- `masked_split(0.1)` — competition-mimic диагностика;
- recall@10: macro (делитель |gt| / min(k,|gt|) / k) и micro — как в `recsys.metrics`.

In [ ]:
import numpy as np
import pandas as pd

events = pd.read_csv(DATA_DIR / "events.csv", dtype={"user_id": "int64", "item_id": "int64", "rating": "int8", "timestamp": "int32"})
catalog_size = pd.read_csv(DATA_DIR / "item_features.csv", usecols=["item_id"])["item_id"].nunique()
print(f"{len(events):,} событий, {events.user_id.nunique():,} пользователей, каталог {catalog_size}")

def chronological_split(events, train_fraction=0.9):
    group = events.groupby("user_id", sort=False)["timestamp"]
    rank, size = group.rank(method="first"), group.transform("size")
    valid_size = np.ceil(size * (1 - train_fraction) - 1e-9).astype(int)
    is_valid = rank > size - valid_size
    return events[~is_valid].reset_index(drop=True), events[is_valid].reset_index(drop=True)

def masked_split(events, valid_fraction=0.1, seed=20261029):
    rng = np.random.default_rng(seed)
    key = rng.random(len(events))
    size = events.groupby("user_id", sort=False)["user_id"].transform("size")
    valid_size = np.ceil(size * valid_fraction - 1e-9).astype(int)
    key_rank = pd.Series(key).groupby(events["user_id"].to_numpy()).rank(method="first", ascending=False)
    is_valid = key_rank.to_numpy() <= valid_size.to_numpy()
    return events[~is_valid].reset_index(drop=True), events[is_valid].reset_index(drop=True)

def recall_panel(recs: dict, valid: pd.DataFrame, k: int = 10) -> dict:
    gt = valid.groupby("user_id")["item_id"].agg(set).to_dict()
    per_user = {u: (len(set(recs.get(u, ())[:k]) & rel), len(rel)) for u, rel in gt.items()}
    h = np.array([x for x, _ in per_user.values()], float)
    n = np.array([y for _, y in per_user.values()], float)
    return {
        "recall@10_macro_relevant": float(np.mean(h / n)),
        "recall@10_macro_capped": float(np.mean(h / np.minimum(k, n))),
        "recall@10_macro_k": float(np.mean(h / k)),
        "recall@10_micro": float(h.sum() / n.sum()),
        "n_eval_users": len(gt),
    }

SPLITS = {"9010": chronological_split, "masked10": masked_split}

## SASRec (RePlay 0.22): обучение и полный скоринг каталога

Конфигурация GPU: embedding 192, 2 блока, 4 головы, окно 200, 40 эпох,
batch 128. Monitor на валидации — recall@10 по первому событию holdout,
итог — наша панель на полном holdout после фильтрации уже оценённых items.

In [ ]:
import lightning as L
import torch
from replay.data.nn import ParquetModule, TensorFeatureInfo, TensorSchema
from replay.data.nn.utils import groupby_sequences
from replay.data.schema import FeatureHint, FeatureType
from replay.nn.lightning import LightningModule
from replay.nn.lightning.callback import ComputeMetricsCallback, PandasTopItemsCallback
from replay.nn.sequential import SasRec
from replay.nn.transform.template import make_default_sasrec_transforms
from pathlib import Path

MAX_SEQ_LEN, EMBEDDING_DIM, NUM_HEADS, NUM_BLOCKS, DROPOUT = 200, 192, 4, 2, 0.3
BATCH_SIZE, EPOCHS, SEED = 128, 40, 42
WORK = Path("gpu_work"); WORK.mkdir(exist_ok=True)

def fit_predict_sasrec(train, valid, epochs=EPOCHS):
    padding = catalog_size
    L.seed_everything(SEED)
    schema = TensorSchema(TensorFeatureInfo(
        "item_id", is_seq=True, padding_value=padding, cardinality=catalog_size,
        embedding_dim=EMBEDDING_DIM, feature_type=FeatureType.CATEGORICAL,
        feature_hint=FeatureHint.ITEM_ID,
    ))
    sequences = groupby_sequences(train[["user_id", "item_id", "timestamp"]], groupby_col="user_id", sort_col="timestamp")
    train_path = WORK / "train.parquet"; sequences.to_parquet(train_path, index=False)
    predict_path = WORK / "predict.parquet"; sequences.to_parquet(predict_path, index=False)

    metadata = {"train": {"item_id": {"shape": MAX_SEQ_LEN + 1, "padding": padding}}}
    callbacks = []
    if valid is not None:
        first_holdout = valid.sort_values(["user_id", "timestamp"]).groupby("user_id")["item_id"].first()
        val_df = sequences[["user_id", "item_id"]].copy()
        val_df["ground_truth"] = val_df["user_id"].map(first_holdout).map(lambda item: [int(item)])
        validate_path = WORK / "validate.parquet"; val_df.to_parquet(validate_path, index=False)
        metadata["validate"] = {"item_id": {"shape": MAX_SEQ_LEN, "padding": padding},
                                "ground_truth": {"shape": 1, "padding": -1}}
        callbacks += [
            ComputeMetricsCallback(metrics=["recall"], ks=[10], item_count=catalog_size),
            L.pytorch.callbacks.ModelCheckpoint(monitor="recall@10", mode="max", dirpath=WORK / "ckpt", filename="best"),
        ]
    transforms = make_default_sasrec_transforms(schema)
    dm = ParquetModule(
        train_path=str(train_path),
        validate_path=str(WORK / "validate.parquet") if valid is not None else None,
        batch_size=BATCH_SIZE, metadata=metadata, transforms=transforms,
    )
    model = SasRec.from_params(schema, embedding_dim=EMBEDDING_DIM, num_heads=NUM_HEADS,
                               num_blocks=NUM_BLOCKS, max_sequence_length=MAX_SEQ_LEN, dropout=DROPOUT)
    module = LightningModule(model)
    trainer = L.Trainer(max_epochs=epochs, accelerator="gpu", devices=1, callbacks=callbacks,
                        logger=True, enable_progress_bar=False, num_sanity_val_steps=0)
    trainer.fit(module, datamodule=dm)

    best_ckpt = WORK / "ckpt" / "best.ckpt"
    if valid is not None and best_ckpt.is_file():
        module = LightningModule.load_from_checkpoint(str(best_ckpt), model=model)
    cb = PandasTopItemsCallback(top_k=catalog_size, query_column="user_id", item_column="item_id", rating_column="score")
    pdm = ParquetModule(predict_path=str(predict_path), batch_size=BATCH_SIZE,
                        metadata={"predict": {"user_id": {}, "item_id": {"shape": MAX_SEQ_LEN, "padding": padding}}},
                        transforms=transforms)
    L.Trainer(accelerator="gpu", devices=1, enable_progress_bar=False, logger=False, inference_mode=True,
              callbacks=[cb]).predict(module, datamodule=pdm, return_predictions=False)
    return cb.get_result().sort_values(["user_id", "score"], ascending=[True, False], kind="stable")

def filter_seen_topk(scores, train, k=10):
    seen = train[["user_id", "item_id"]].drop_duplicates()
    unseen = scores.merge(seen, on=["user_id", "item_id"], how="left", indicator=True)
    unseen = unseen[unseen["_merge"] == "left_only"].drop(columns="_merge")
    top = unseen.groupby("user_id", sort=False).head(k).reset_index(drop=True)
    top["rank"] = top.groupby("user_id", sort=False).cumcount() + 1
    return top

print("pipeline готов")

In [ ]:
import json
out_dir = Path("gpu_outputs"); out_dir.mkdir(exist_ok=True)
payload = {"splits": {}, "results": {}}
for split_name, splitter in SPLITS.items():
    train, valid = splitter(events)
    payload["splits"][split_name] = {"train": len(train), "valid": len(valid)}
    print(f"=== {split_name} ===", flush=True)
    scores = fit_predict_sasrec(train, valid)
    recs = filter_seen_topk(scores, train)
    recs_map = recs.groupby("user_id")["item_id"].apply(list).to_dict()
    panel = recall_panel(recs_map, valid)
    payload["results"][split_name] = {"sasrec_gpu_e192_len200_ep40": {k: round(v, 6) for k, v in panel.items()}}
    print(split_name, payload["results"][split_name], flush=True)
(out_dir / "results_e05b.json").write_text(json.dumps(payload, indent=2))
print("результаты:", out_dir / "results_e05b.json")

## Сабмит на полных events

Обучение без валидации (всё выданное — прошлое скрытого теста), те же эпохи.

In [ ]:
scores = fit_predict_sasrec(events, None)
recs = filter_seen_topk(scores, events)
top = recs.groupby("user_id")["item_id"].apply(lambda x: " ".join(map(str, x))).reset_index()
top.columns = ["user_id", "item_id"]
top.sort_values("user_id").to_csv(out_dir / "E05b_sasrec.csv", index=False)
print(f"сабмит: {out_dir / 'E05b_sasrec.csv'}, пользователей: {len(top)}")
assert len(top) == 6040 and top['item_id'].str.split().str.len().eq(10).all()